# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

1. Impotar librerías

In [68]:
import os
import re
import numpy as np
import string
import nltk
from nltk.corpus import stopwords

2. Definir variables
    - *path:* Ruta del archivo
    - *corpus:* Colección de documentos
    - *vocabulary:* Conjunto de palabras únicas

In [74]:
path = "01_corpus_turismo_500.txt"
corpus = {}
vocabulary = set()

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

3. Se lee el archivo, tratando cada línea del archivo "01_corpus_turismo_500.txt" como un documento. Posteriormente se realiza una limpieza y tokenización de los documentos. Se guarda el contenido del documento como una lista de palabras y se actualiza el vocabulario.

In [75]:
with open(path, "r", encoding="utf-8") as f:
    for n, line in enumerate(f):
        sentence = line.strip('.\n,')
        if sentence:
            sentence = sentence.lower()
            # Eliminar numeros
            clean_text = re.sub(r'\d+', '', sentence)
            # Eliminar signos de puntuación
            clean_text = clean_text.translate(str.maketrans('', '', string.punctuation))
            tokens = clean_text.split()
            stop_words = set(stopwords.words('spanish'))
            clean_tokens  = [word for word in tokens if word not in stop_words]
            corpus[n] = clean_tokens
            vocabulary.update(clean_tokens)

print(f"Cantidad de documentos: {len(corpus)}")
print(f"Cantidad de palabras en el vocabulario: {len(vocabulary)}")

Cantidad de documentos: 500
Cantidad de palabras en el vocabulario: 95


4. Se construye un índice del vocabulario para acceder a la ubicación de la palabra en la matriz. Además, se crea la matriz "frequency_matrix", en donde se cuenta las apariciones de cada palabra del vocabulario dentro de los documentos.

In [ ]:
def create_tf_matrix(corpus: dict, vocabulary: set)-> np.ndarray:
    vocabulary = sorted(vocabulary)
    index_vocabulary = {word : i for i, word in enumerate(vocabulary)}
    term_frequency_matrix = np.zeros((len(corpus), len(vocabulary)), dtype = np.float32)
    for num_doc, content in enumerate(corpus.values()):
        for word in content:
            index_word = index_vocabulary[word]
            term_frequency_matrix[num_doc][index_word] += 1
    mask = term_frequency_matrix > 0
    term_frequency_matrix[mask] = 1 + np.log(term_frequency_matrix[mask])
    return term_frequency_matrix

Se invoca a la función pasandole el corpus y el vocabulario

In [76]:
tf_matrix = create_tf_matrix(corpus, vocabulary)
print(f"TF matrix shape: {tf_matrix.shape}")

TF matrix shape: (500, 95)


5. Se calcula el vector idf, el cual expresa que tan informativo es un término, con la siguiente fórmula: $idf_t = \log \left( \frac{N}{df_t} \right)$
Por último, se multiplica la matriz de frecuencia de términos, con el vector idf.



In [63]:
def calculate_tf_idf(tf_matrix: np.ndarray, len_corpus: int)-> tuple[np.ndarray, np.ndarray]:
    document_frequency_vector = np.sum(tf_matrix > 0, axis=0)
    idf_vector = np.log(len_corpus / (document_frequency_vector+1)).astype(np.float32)
    tf_idf_matrix = tf_matrix * idf_vector
    tf_idf_matrix = tf_idf_matrix / np.linalg.norm(tf_idf_matrix, axis=1, keepdims=True)
    return tf_idf_matrix, idf_vector

Se invoca a la función pasandole la matriz de frecuencia de términos y el tamaño del corpus

In [ ]:
tf_idf_matrix, idf_vector = calculate_tf_idf(tf_matrix, len(corpus))
print(f"TF-IDF matrix shape: {tf_idf_matrix.shape}")

TF-IDF matrix shape: (500, 95)
IDF vector shape: (95,)


### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

1. Se definen las mismas variables del anterior ejercicio, sumando una nueva:
    - *files:* lista de nombres de los archivos dentro de un directorio.

In [26]:
path = 'C:/Users/david/Documents/7mo semestre/Recuperacion Informacion/ir26a/100libros/'
files = os.listdir(path)

vocabulary = set()
corpus = {}

start_pattern = r'\*\*\* START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*'

2. Se recorre la lista de archivos de files. Se hace una limpieza con la librería 're' de signos de puntuación, además de volver en minúsculas a todo el texto. Se guarda el contenido como una lista de palabras en el diccionario 'corpus' y se actualiza el vocabulario con cada iteración.

In [33]:
def preprocess_text(text: str)-> list:
    partes = re.split(start_pattern, text)
    if len(partes) > 1:
        text = partes[1]
    lower_text = text.lower()
    clean_text = re.sub(r'\d+', '', lower_text)
    clean_text = clean_text.translate(str.maketrans('', '', string.punctuation))
    tokens = clean_text.split()
    stop_words = set(stopwords.words('spanish'))
    clean_tokens  = [word for word in tokens if word not in stop_words]
    return clean_tokens

In [34]:
for doc in files:
    with open(path + doc, 'r', encoding= 'utf-8') as f:
        text = f.read()
        tokens = preprocess_text(text)       
        vocabulary.update(tokens)
        corpus[doc] = tokens

print(f"Cantidad de documentos: {len(corpus)}")
print(f"Cantidad de palabras en el vocabulario: {len(vocabulary)}")

Cantidad de documentos: 907
Cantidad de palabras en el vocabulario: 952875


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

1. Al utilizar los mismos cálculos y lógica que el primer ejercicio, se utilizó la función definida al inicio de este notebook.

In [64]:
tf_matrix = create_tf_matrix(corpus, vocabulary)
print(f"TF matrix shape: {tf_matrix.shape}")

TF matrix shape: (907, 952875)


In [65]:
(tf_idf_matrix, idf_vector) = calculate_tf_idf(tf_matrix, len(corpus)) 
print(f"TF-IDF matrix shape: {tf_idf_matrix.shape}")

TF-IDF matrix shape: (907, 952875)


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

1. En la función se realizan las siguientes acciones:
    - Se limpia el query, se cambian a minúsculas y se separan en palabras
    - Se prepara el vector de query para medir la frecuencia de las palabras con la misma longitud que el vocabulario.
    - Se contabilizan las repeticiones ubican las pocisiones en el vector mediante el indice del vocabulario.
    - Se comprueba que la norma del vector no sea 0, caso contrario devuelve la búsqueda como vacía.
    - Se normaliza el vector del query
    - Se aplica el producto punto para obtener el vector de similitudes
    - Se ordena de forma descendente y se devuelven los 5 mejores resultados

In [58]:
index_vocabulary = {word : i for i, word in enumerate(vocabulary)}  
documento = {num: doc for num, doc in enumerate(corpus.keys())}

def search(query, tf_idf_matrix, idf_vector):
    tokens = preprocess_text(query)
    vector_query = np.zeros((len(vocabulary),), dtype=int)
    for word in tokens:
        if word in index_vocabulary:
            index_word = index_vocabulary[word]
            vector_query[index_word] += 1

    
    vector_query = vector_query * idf_vector
    norma_query = np.linalg.norm(vector_query)
    
    if norma_query == 0:
        return []

    vector_query_normalizado = vector_query / norma_query
    similitudes = tf_idf_matrix @ vector_query_normalizado
    indices_ordenados = np.argsort(similitudes)[::-1]
    resultados = [documento[i] for i in indices_ordenados[:5]]
    
    return resultados

In [66]:
print(search('después haya de arrepentirse', tf_idf_matrix, idf_vector))

['Insolación y Morriña (Dos historias amorosas).txt', 'Fortunata y Jacinta dos historias de casadas.txt', 'Zaragoza.txt', 'Zalacaín El Aventurero (Historia de las buenas andanzas y fortunas de Martín Zalacaín el Aventurero).txt', 'Zadig, ó El Destino, Historia Oriental.txt']
